# Comprehensive Model Training Notebook

This notebook trains **ALL** models for the MetaTrader Multi-Strategy EA:

## PyTorch Models (ONNX Export for MT5)
- **SequenceMLP** - Baseline MLP for MT5 ONNX runtime
- **PatchTST** - Patch Time Series Transformer for sequence modeling
- **iTransformer** - Inverted Transformer (attention over features)

## Tree-Based Models (Per Asset Class)
- **Forex** → LightGBM (smooth signals, low learning rate)
- **Metals (Gold/Silver)** → CatBoost + XGBoost (breakout detection)
- **Indices (US30, US100, etc.)** → XGBoost (mean-reversion)
- **Energies (WTI, Brent, NG)** → XGBoost (volatility breakout)

## Deriv Synthetic Indices (18 Families)
- CatBoost, LightGBM, XGBoost per family (0-17)
- Family-specific hyperparameters
- Extended sequences for Jump (3) and DEX (4) families

## Stacker / Ensemble
- Ridge regression stacker on OOF predictions
- Combines ONNX + LightGBM + CatBoost + XGBoost

## Validation & Export
- CPCV (Combinatorial Purged Cross-Validation)
- ONNX export for MT5 runtime
- Deployment gate: IC > 0.02 + CPCV pass

## Usage
1. Set `DATA_CSV` path to your training data (H1 recommended)
2. Set `OUTPUT_DIR` for model outputs
3. Run cells sequentially
4. Models exported to `OUTPUT_DIR` as `.onnx` (PyTorch) and `.pkl` (tree-based)

## Prerequisites
```bash
pip install torch numpy pandas scipy scikit-learn onnx onnxruntime lightgbm catboost xgboost optuna matplotlib seaborn
```

## 1. Setup & Configuration

In [ ]:
# ============================================================
# Cell 1 — Setup & Configuration
# ============================================================

import os
import sys
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
import json
import pickle

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

# Import project modules
from data_pipeline import (
    build_scaled_dataset_splits,
    PipelineMetadata,
    TradingDataset,
    build_feature_matrix,
    get_feature_count,
)
from models import SequenceMLP, PatchTST, iTransformer
from train_model import (
    train_candidate,
    compute_ic,
    export_onnx,
    instantiate_models,
    build_loader,
)
from validate_model import run_cpcv

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# Configuration
# ============================================================
@dataclass
class TrainingConfig:
    # Data paths
    data_csv: str = "../Python/data/AITraining_OHLCV_H1.csv"  # H1 data recommended
    output_dir: str = "../Resources"
    
    # Model selection - set to True/False to enable/disable
    train_pytorch_models: bool = True
    pytorch_model_types: List[str] = field(default_factory=lambda: ["mlp", "patchtst", "itransformer"])
    
    train_asset_class_models: bool = True
    train_deriv_family_models: bool = True
    train_stacker: bool = True
    
    # Architecture (PyTorch models)
    seq_len: int = 60
    n_features: int = 57
    n_classes: int = 3
    
    # Triple-barrier labeling
    tb_k: float = 1.5
    tb_vertical_bars: int = 20
    
    # Training hyperparameters (PyTorch)
    epochs: int = 60
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 1e-4
    patience: int = 20
    min_ic: float = 0.02
    
    # CPCV
    cpcv_splits: int = 6
    
    # Deriv family (set -1 for universal)
    family_id: int = -1
    
    # Asset class (0=forex, 1=metals, 2=indices, 3=energies, -1=universal)
    asset_class: int = -1
    
    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Reproducibility
    seed: int = 42
    
    # Force export even if gate fails
    force_export: bool = False

cfg = TrainingConfig()

# Set random seeds
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

# Setup output directory
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\nConfiguration:")
print(f"  Data: {cfg.data_csv}")
print(f"  Output: {cfg.output_dir}")
print(f"  Device: {cfg.device}")
print(f"  PyTorch models: {cfg.train_pytorch_models} ({cfg.pytorch_model_types})")
print(f"  Asset class models: {cfg.train_asset_class_models}")
print(f"  Deriv family models: {cfg.train_deriv_family_models}")
print(f"  Stacker: {cfg.train_stacker}")

## 2. Data Loading & Inspection

In [ ]:
# ============================================================
# Cell 2 — Data Loading & Inspection
# ============================================================

data_path = Path(cfg.data_csv)
if not data_path.exists():
    # Try common locations
    possible_paths = [
        "../Python/data/AITraining_OHLCV_H1.csv",
        "../Python/data/AITraining_OHLCV_M30.csv",
        "../Python/data/AITraining_OHLCV_M15.csv",
        "../Python/data/UNIVERSAL_TRAINING_SET.csv",
        "data/AITraining_OHLCV_H1.csv",
    ]
    for path in possible_paths:
        if Path(path).exists():
            data_path = Path(path)
            cfg.data_csv = str(data_path)
            break
    
if not data_path.exists():
    raise FileNotFoundError(
        f"Training data not found. Tried: {cfg.data_csv}"
    )

print(f"Loading data from: {data_path}")

# Quick data inspection (sample first 1000 rows)
df_sample = pd.read_csv(data_path, nrows=1000)
print(f"\nSample shape: {df_sample.shape}")
print(f"Columns: {list(df_sample.columns[:10])}{'...' if len(df_sample.columns) > 10 else ''}")

if 'date' in df_sample.columns:
    df_sample['date'] = pd.to_datetime(df_sample['date'])
    print(f"Date range (sample): {df_sample['date'].min()} → {df_sample['date'].max()}")

if 'symbol' in df_sample.columns:
    print(f"Symbols in sample: {df_sample['symbol'].nunique()}")
    print(f"Symbol counts:\n{df_sample['symbol'].value_counts().head(20).to_string()}")

# Full data stats (this may take a moment for large files)
print("\nReading full dataset for stats...")
df = pd.read_csv(data_path, usecols=['symbol', 'date'])
df['date'] = pd.to_datetime(df['date'])
print(f"Full dataset:")
print(f"  Rows: {len(df):,}")
print(f"  Symbols: {df['symbol'].nunique()}")
print(f"  Date range: {df['date'].min()} → {df['date'].max()}")
print(f"  Symbol counts:\n{df['symbol'].value_counts().to_string()}")

## 3. PyTorch Model Training (ONNX Export)

In [ ]:
# ============================================================
# Cell 3 — PyTorch Model Training
# ============================================================

if cfg.train_pytorch_models:
    print("="*60)
    print("PYTORCH MODEL TRAINING")
    print("="*60)

    # Build dataset splits for universal model (family_id=-1)
    seq_len = cfg.seq_len
    if cfg.family_id in (3, 4) and seq_len == 60:
        seq_len = 120
        print(f"Adjusted sequence length to {seq_len} for family_id {cfg.family_id}")

    scaler_output = str(output_dir / "scaler.bin")

    print(f"\nBuilding dataset splits...")
    print(f"  Sequence length: {seq_len}")
    print(f"  Triple-barrier k: {cfg.tb_k}")
    print(f"  Vertical bars: {cfg.tb_vertical_bars}")
    print(f"  Family ID: {cfg.family_id}")

    train_split, val_split, test_split, metadata = build_scaled_dataset_splits(
        cfg.data_csv,
        seq_len=seq_len,
        k=cfg.tb_k,
        vertical_bars=cfg.tb_vertical_bars,
        scaler_output=scaler_output,
        family_id=cfg.family_id,
        asset_class=cfg.asset_class,
    )

    print(f"\nDataset splits:")
    print(f"  Train: {metadata.train_size:,} samples")
    print(f"  Val:   {metadata.val_size:,} samples")
    print(f"  Test:  {metadata.test_size:,} samples")
    print(f"  Features: {metadata.n_features}")
    print(f"  Sequence length: {metadata.seq_len}")
    print(f"  Annualization factor: {metadata.annualization:.1f}")

    # Check class distribution
    y_train = train_split[1]
    class_counts = np.bincount(y_train, minlength=3)
    print(f"\nClass distribution (train):")
    print(f"  SELL (0):     {class_counts[0]:,} ({class_counts[0]/len(y_train)*100:.1f}%)")
    print(f"  NEUTRAL (1):  {class_counts[1]:,} ({class_counts[1]/len(y_train)*100:.1f}%)")
    print(f"  BUY (2):      {class_counts[2]:,} ({class_counts[2]/len(y_train)*100:.1f}%)")

    # Train each model type
    results = {}
    for model_type in cfg.pytorch_model_types:
        print(f"\n{'='*60}")
        print(f"TRAINING: {model_type.upper()}")
        print(f"{'='*60}")
        
        model = instantiate_models(model_type, metadata.seq_len, metadata.n_features)[model_type]
        print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

        # Test forward pass
        dummy_input = torch.zeros(1, metadata.seq_len, metadata.n_features)
        with torch.no_grad():
            dummy_output = model(dummy_input)
        print(f"Input shape:  {tuple(dummy_input.shape)}")
        print(f"Output shape: {tuple(dummy_output.shape)}")

        # Train
        val_ic, test_ic, trained_model = train_candidate(
            name=model_type,
            model=model,
            train_split=train_split,
            val_split=val_split,
            test_split=test_split,
            metadata=metadata,
            epochs=cfg.epochs,
            batch_size=cfg.batch_size,
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
            device=cfg.device,
        )

        # Export ONNX
        temp_output = str(output_dir / f"{model_type}.tmp.onnx")
        export_onnx(
            trained_model,
            metadata.seq_len,
            metadata.n_features,
            temp_output,
            opset=12,
        )

        # CPCV Validation
        print(f"\nRunning CPCV validation ({cfg.cpcv_splits} splits)...")
        X_all = np.concatenate([train_split[0], val_split[0], test_split[0]], axis=0)
        y_all = np.concatenate([train_split[1], val_split[1], test_split[1]], axis=0)
        returns_all = np.concatenate([train_split[3], val_split[3], test_split[3]], axis=0)
        
        cpcv_result = run_cpcv(
            model_path=temp_output,
            X=X_all,
            y=y_all,
            bar_returns=returns_all,
            n_splits=cfg.cpcv_splits,
            annualization=metadata.annualization,
        )

        # Deployment gate
        deploy_ok = test_ic >= cfg.min_ic and cpcv_result['deploy_gate']
        
        print(f"\n--- {model_type.upper()} RESULTS ---")
        print(f"  Val IC:     {val_ic:.4f}")
        print(f"  Test IC:    {test_ic:.4f} (min: {cfg.min_ic})")
        print(f"  CPCV Sharpe: {cpcv_result['mean_sr']:.4f}")
        print(f"  CPCV P10:   {cpcv_result['p10_sr']:.4f}")
        print(f"  CPCV PSR:   {cpcv_result['psr']:.4f}")
        print(f"  CPCV Pass:  {cpcv_result['deploy_gate']}")
        print(f"  Deploy:     {'✅ APPROVED' if deploy_ok else '❌ REJECTED'}")

        if deploy_ok or cfg.force_export:
            final_output = str(output_dir / f"{model_type}.onnx")
            Path(temp_output).rename(final_output)
            print(f"  ✅ Exported to: {final_output}")
        else:
            Path(temp_output).unlink(missing_ok=True)
            print(f"  ❌ Export skipped (gate failed)")

        results[model_type] = {
            'val_ic': val_ic,
            'test_ic': test_ic,
            'cpcv': cpcv_result,
            'deploy_ok': deploy_ok,
            'model_path': final_output if deploy_ok or cfg.force_export else None
        }

    # Summary
    print(f"\n{'='*60}")
    print("PYTORCH TRAINING SUMMARY")
    print(f"{'='*60}")
    for model_type, res in results.items():
        status = "✅" if res['deploy_ok'] else "❌"
        print(f"  {model_type:12s} | Val IC: {res['val_ic']:.4f} | Test IC: {res['test_ic']:.4f} | CPCV: {res['cpcv']['mean_sr']:.4f} | {status}")

    # Save results
    with open(output_dir / "pytorch_training_results.json", "w") as f:
        json.dump({k: {kk: (float(vv) if isinstance(vv, (np.floating, np.integer)) else vv) 
                       for kk, vv in v.items() if kk != 'cpcv'}
                  for k, v in results.items()}, f, indent=2)
else:
    print("PyTorch training disabled in config.")
    results = {}

## 4. Asset Class Model Training (Tree-Based)

In [ ]:
# ============================================================
# Cell 4 — Asset Class Model Training
# ============================================================

asset_class_results = {}

if cfg.train_asset_class_models:
    print("="*60)
    print("ASSET CLASS MODEL TRAINING (Tree-Based)")
    print("="*60)

    # Asset class configurations
    asset_classes = [
        {"id": 0, "name": "forex", "models": ["lgbm"]},
        {"id": 1, "name": "metals", "models": ["catboost", "xgboost"]},
        {"id": 2, "name": "indices", "models": ["xgboost"]},
        {"id": 3, "name": "energies", "models": ["xgboost"]},
    ]

    for ac in asset_classes:
        print(f"\n{'='*60}")
        print(f"TRAINING: {ac['name'].upper()} (asset_class={ac['id']})")
        print(f"{'='*60}")
        
        # Build dataset for this asset class
        train_split, val_split, test_split, metadata = build_scaled_dataset_splits(
            cfg.data_csv,
            seq_len=cfg.seq_len,
            k=cfg.tb_k,
            vertical_bars=cfg.tb_vertical_bars,
            family_id=-1,
            asset_class=ac['id'],
        )

        print(f"  Dataset: Train={metadata.train_size:,}, Val={metadata.val_size:,}, Test={metadata.test_size:,}")
        print(f"  Features: {metadata.n_features}")

        X_tr = train_split[0].reshape(len(train_split[0]), -1)
        X_va = val_split[0].reshape(len(val_split[0]), -1)
        X_te = test_split[0].reshape(len(test_split[0]), -1)
        y_tr, y_va, y_te = train_split[1], val_split[1], test_split[1]
        r_te = test_split[3]

        asset_results = {}
        
        for model_type in ac["models"]:
            print(f"\n  --- Training {model_type.upper()} ---")
            
            if model_type == "lgbm":
                try:
                    import lightgbm as lgb
                except ImportError:
                    print("  ⚠️  LightGBM not installed, skipping")
                    continue
                
                if ac['name'] == 'forex':
                    params = {
                        "objective": "multiclass",
                        "num_class": 3,
                        "metric": "multi_logloss",
                        "learning_rate": 0.025,
                        "num_leaves": 31,
                        "feature_fraction": 0.7,
                        "bagging_fraction": 0.8,
                        "bagging_freq": 5,
                        "lambda_l1": 0.1,
                        "lambda_l2": 0.1,
                        "min_child_samples": 50,
                        "verbose": -1,
                    }
                    num_boost_round = 800
                    early_stopping = 50
                else:
                    params = {
                        "objective": "multiclass",
                        "num_class": 3,
                        "metric": "multi_logloss",
                        "learning_rate": 0.03,
                        "num_leaves": 63,
                        "feature_fraction": 0.7,
                        "bagging_fraction": 0.8,
                        "bagging_freq": 5,
                        "lambda_l1": 0.1,
                        "lambda_l2": 0.1,
                        "min_child_samples": 50,
                        "verbose": -1,
                    }
                    num_boost_round = 1000
                    early_stopping = 50
                
                model = lgb.train(
                    params,
                    lgb.Dataset(X_tr, label=y_tr),
                    valid_sets=[lgb.Dataset(X_va, label=y_va)],
                    num_boost_round=num_boost_round,
                    callbacks=[lgb.early_stopping(early_stopping)],
                )
                
                pkl_output = output_dir / f"{ac['name']}_lgbm.pkl"
                with open(pkl_output, "wb") as f:
                    pickle.dump(model, f)
                
                preds = model.predict(X_te)
                test_ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)
                
            elif model_type == "catboost":
                try:
                    import catboost as cb
                except ImportError:
                    print("  ⚠️  CatBoost not installed, skipping")
                    continue
                
                if ac['name'] == 'metals':
                    model = cb.CatBoostClassifier(
                        iterations=600,
                        learning_rate=0.03,
                        depth=6,
                        l2_leaf_reg=3.0,
                        loss_function="MultiClass",
                        classes_count=3,
                        verbose=0,
                        early_stopping_rounds=50,
                    )
                else:
                    model = cb.CatBoostClassifier(
                        iterations=1000,
                        learning_rate=0.03,
                        depth=6,
                        l2_leaf_reg=3.0,
                        loss_function="MultiClass",
                        classes_count=3,
                        verbose=0,
                        early_stopping_rounds=50,
                    )
                
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
                
                pkl_output = output_dir / f"{ac['name']}_catboost.pkl"
                with open(pkl_output, "wb") as f:
                    pickle.dump(model, f)
                
                preds = model.predict_proba(X_te)
                test_ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)
                
            elif model_type == "xgboost":
                try:
                    import xgboost as xgb
                except ImportError:
                    print("  ⚠️  XGBoost not installed, skipping")
                    continue
                
                if ac['name'] == 'metals':
                    model = xgb.XGBClassifier(
                        n_estimators=500,
                        learning_rate=0.03,
                        max_depth=6,
                        subsample=0.8,
                        colsample_bytree=0.7,
                        objective="multi:softprob",
                        num_class=3,
                        eval_metric="mlogloss",
                        early_stopping_rounds=50,
                        verbosity=0,
                    )
                elif ac['name'] == 'indices':
                    model = xgb.XGBClassifier(
                        n_estimators=600,
                        learning_rate=0.025,
                        max_depth=5,
                        subsample=0.8,
                        colsample_bytree=0.7,
                        objective="multi:softprob",
                        num_class=3,
                        eval_metric="mlogloss",
                        early_stopping_rounds=50,
                        verbosity=0,
                    )
                elif ac['name'] == 'energies':
                    model = xgb.XGBClassifier(
                        n_estimators=500,
                        learning_rate=0.03,
                        max_depth=6,
                        subsample=0.8,
                        colsample_bytree=0.7,
                        objective="multi:softprob",
                        num_class=3,
                        eval_metric="mlogloss",
                        early_stopping_rounds=50,
                        verbosity=0,
                    )
                
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
                
                pkl_output = output_dir / f"{ac['name']}_xgboost.pkl"
                with open(pkl_output, "wb") as f:
                    pickle.dump(model, f)
                
                preds = model.predict_proba(X_te)
                test_ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)

            asset_results[model_type] = {
                'test_ic': float(test_ic),
                'model_path': str(pkl_output),
            }
            print(f"    Test IC: {test_ic:.4f}")
            print(f"    Saved:   {pkl_output}")

        asset_class_results[ac['name']] = asset_results

    # Summary
    print(f"\n{'='*60}")
    print("ASSET CLASS TRAINING SUMMARY")
    print(f"{'='*60}")
    for ac_name, models in asset_class_results.items():
        print(f"\n  {ac_name.upper()}:")
        for model_type, res in models.items():
            print(f"    {model_type:10s} | Test IC: {res['test_ic']:.4f} | {Path(res['model_path']).name}")

    # Save results
    with open(output_dir / "asset_class_training_results.json", "w") as f:
        json.dump(asset_class_results, f, indent=2)

else:
    print("Asset class training disabled in config.")

## 5. Deriv Family Model Training (18 Families)

In [ ]:
# ============================================================
# Cell 5 — Deriv Family Model Training
# ============================================================

FAMILY_NAMES = [
    "crashboom", "volatility", "step", "jump", "dex",
    "multistep", "exponential", "hybrid", "rangebreak",
    "skewstep", "volswitch", "driftswitch", "trek",
    "tactical", "derived", "stablespread", "pairsarbitrage", "spotvolatility",
]

deriv_results = {}

if cfg.train_deriv_family_models:
    print("="*60)
    print("DERIV FAMILY MODEL TRAINING (18 Families)")
    print("="*60)

    # Which models to train per family (can customize per family)
    family_model_config = {
        0: ["catboost"],           # CrashBoom
        1: ["lgbm"],               # Volatility
        2: ["xgboost"],            # Step
        3: ["catboost", "lgbm"],  # Jump (seq_len=120)
        4: ["catboost", "lgbm"],  # DEX (seq_len=120)
        5: ["xgboost"],            # MultiStep
        6: ["catboost"],           # Exponential
        7: ["catboost", "lgbm"],  # Hybrid
        8: ["xgboost"],            # RangeBreak
        9: ["xgboost"],            # SkewStep
        10: ["catboost"],          # VolSwitch
        11: ["catboost"],         # DriftSwitch
        12: ["xgboost"],           # Trek
        13: ["catboost"],          # Tactical
        14: ["xgboost"],           # Derived
        15: ["catboost"],          # StableSpread
        16: ["catboost"],          # PairsArbitrage
        17: ["xgboost"],           # SpotVolatility
    }

    for family_id in range(18):
        family_name = FAMILY_NAMES[family_id]
        models_to_train = family_model_config.get(family_id, ["catboost"])
        
        # Adjust sequence length for Jump (3) and DEX (4)
        seq_len = cfg.seq_len
        if family_id in (3, 4) and seq_len == 60:
            seq_len = 120

        print(f"\n{'='*60}")
        print(f"FAMILY {family_id:2d}: {family_name.upper()} (seq_len={seq_len})")
        print(f"{'='*60}")
        
        # Build dataset for this family
        train_split, val_split, test_split, metadata = build_scaled_dataset_splits(
            cfg.data_csv,
            seq_len=seq_len,
            k=cfg.tb_k,
            vertical_bars=cfg.tb_vertical_bars,
            family_id=family_id,
        )

        print(f"  Dataset: Train={metadata.train_size:,}, Val={metadata.val_size:,}, Test={metadata.test_size:,}")
        print(f"  Features: {metadata.n_features}")

        if metadata.train_size < 100:
            print(f"  ⚠️  Insufficient data ({metadata.train_size} samples), skipping")
            continue

        X_tr = train_split[0].reshape(len(train_split[0]), -1)
        X_va = val_split[0].reshape(len(val_split[0]), -1)
        X_te = test_split[0].reshape(len(test_split[0]), -1)
        y_tr, y_va, y_te = train_split[1], val_split[1], test_split[1]
        r_te = test_split[3]

        family_results = {}
        
        for model_type in models_to_train:
            print(f"\n  --- Training {model_type.upper()} ---")
            
            if model_type == "catboost":
                try:
                    import catboost as cb
                except ImportError:
                    print("  ⚠️  CatBoost not installed, skipping")
                    continue
                
                # Family-specific hyperparameters
                if family_id in (0, 4):  # CrashBoom, DEX
                    model = cb.CatBoostClassifier(
                        iterations=1500,
                        learning_rate=0.025,
                        depth=8,
                        loss_function="MultiClass",
                        classes_count=3,
                        l2_leaf_reg=5.0,
                        random_seed=42,
                        verbose=100,
                        early_stopping_rounds=75,
                        class_weights=[1.0, 0.5, 1.0],
                    )
                elif family_id == 7:  # Hybrid
                    model = cb.CatBoostClassifier(
                        iterations=1200,
                        learning_rate=0.03,
                        depth=7,
                        loss_function="MultiClass",
                        classes_count=3,
                        l2_leaf_reg=4.0,
                        random_seed=42,
                        verbose=100,
                        early_stopping_rounds=50,
                    )
                else:  # Default
                    model = cb.CatBoostClassifier(
                        iterations=1000,
                        learning_rate=0.03,
                        depth=6,
                        loss_function="MultiClass",
                        classes_count=3,
                        l2_leaf_reg=3.0,
                        random_seed=42,
                        verbose=100,
                        early_stopping_rounds=50,
                    )
                
                from catboost import Pool
                train_pool = Pool(X_tr, label=y_tr)
                val_pool = Pool(X_va, label=y_va)
                model.fit(train_pool, eval_set=val_pool)
                
                pkl_output = output_dir / f"{family_name}_catboost.pkl"
                with open(pkl_output, "wb") as f:
                    pickle.dump(model, f)
                
                preds = model.predict_proba(X_te)
                test_ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)
                
            elif model_type == "lgbm":
                try:
                    import lightgbm as lgb
                except ImportError:
                    print("  ⚠️  LightGBM not installed, skipping")
                    continue
                
                if family_id == 1:  # Volatility
                    params = {
                        "objective": "multiclass",
                        "num_class": 3,
                        "metric": "multi_logloss",
                        "learning_rate": 0.02,
                        "num_leaves": 31,
                        "feature_fraction": 0.7,
                        "bagging_fraction": 0.8,
                        "bagging_freq": 5,
                        "lambda_l1": 0.1,
                        "lambda_l2": 0.1,
                        "min_child_samples": 50,
                        "verbose": -1,
                    }
                else:  # Default
                    params = {
                        "objective": "multiclass",
                        "num_class": 3,
                        "metric": "multi_logloss",
                        "learning_rate": 0.03,
                        "num_leaves": 63,
                        "feature_fraction": 0.7,
                        "bagging_fraction": 0.8,
                        "bagging_freq": 5,
                        "lambda_l1": 0.1,
                        "lambda_l2": 0.1,
                        "min_child_samples": 50,
                        "verbose": -1,
                    }
                
                model = lgb.train(
                    params,
                    lgb.Dataset(X_tr, label=y_tr),
                    valid_sets=[lgb.Dataset(X_va, label=y_va)],
                    num_boost_round=1000,
                    callbacks=[lgb.early_stopping(50)],
                )
                
                pkl_output = output_dir / f"{family_name}_lgbm.pkl"
                with open(pkl_output, "wb") as f:
                    pickle.dump(model, f)
                
                preds = model.predict(X_te)
                test_ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)
                
            elif model_type == "xgboost":
                try:
                    import xgboost as xgb
                except ImportError:
                    print("  ⚠️  XGBoost not installed, skipping")
                    continue
                
                if family_id in (2, 5, 9):  # Step, MultiStep, SkewStep
                    model = xgb.XGBClassifier(
                        objective="multi:softprob",
                        num_class=3,
                        learning_rate=0.03,
                        max_depth=6,
                        n_estimators=1000,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        gamma=1.0,
                        reg_alpha=0.5,
                        reg_lambda=2.0,
                        min_child_weight=50,
                        early_stopping_rounds=50,
                    )
                else:  # Default
                    model = xgb.XGBClassifier(
                        objective="multi:softprob",
                        num_class=3,
                        learning_rate=0.03,
                        max_depth=6,
                        n_estimators=1000,
                        subsample=0.8,
                        colsample_bytree=0.7,
                        reg_alpha=0.1,
                        reg_lambda=0.1,
                        min_child_weight=50,
                        early_stopping_rounds=50,
                        verbosity=0,
                    )
                
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=100)
                
                pkl_output = output_dir / f"{family_name}_xgboost.pkl"
                with open(pkl_output, "wb") as f:
                    pickle.dump(model, f)
                
                preds = model.predict_proba(X_te)
                test_ic, _ = spearmanr(preds[:, 2] - preds[:, 0], r_te)

            family_results[model_type] = {
                'test_ic': float(test_ic),
                'model_path': str(pkl_output),
            }
            print(f"    Test IC: {test_ic:.4f}")
            print(f"    Saved:   {pkl_output}")

        deriv_results[family_name] = {
            'family_id': family_id,
            'seq_len': seq_len,
            'models': family_results,
            'dataset_size': metadata.train_size,
        }

    # Summary
    print(f"\n{'='*60}")
    print("DERIV FAMILY TRAINING SUMMARY")
    print(f"{'='*60}")
    for family_name, res in deriv_results.items():
        print(f"\n  {family_name} (id={res['family_id']}, seq_len={res['seq_len']}, n={res['dataset_size']:,}):")
        for model_type, mres in res['models'].items():
            print(f"    {model_type:10s} | Test IC: {mres['test_ic']:.4f}")

    # Save results
    with open(output_dir / "deriv_family_training_results.json", "w") as f:
        json.dump(deriv_results, f, indent=2)

else:
    print("Deriv family training disabled in config.")

## 6. Stacker / Ensemble Training

In [ ]:
# ============================================================
# Cell 6 — Stacker / Ensemble Training
# ============================================================

stacker_results = {}

if cfg.train_stacker:
    print("="*60)
    print("STACKER / ENSEMBLE TRAINING")
    print("="*60)

    try:
        import onnxruntime as ort
        from sklearn.linear_model import Ridge
        from sklearn.preprocessing import StandardScaler
        from sklearn.model_selection import TimeSeriesSplit
        from scipy.special import softmax
    except ImportError as e:
        print(f"\u26a0️  Missing dependencies for stacker: {e}")
        cfg.train_stacker = False

if cfg.train_stacker:
    # Train stacker for each family that has base models
    for family_id in range(18):
        family_name = FAMILY_NAMES[family_id]
        
        # Check which base models exist
        onnx_path = output_dir / "patchtst.onnx"  # Universal ONNX model
        lgbm_path = output_dir / f"{family_name}_lgbm.pkl"
        catboost_path = output_dir / f"{family_name}_catboost.pkl"
        xgboost_path = output_dir / f"{family_name}_xgboost.pkl"

        # Need at least ONNX + LGBM for stacker
        if not onnx_path.exists() or not lgbm_path.exists():
            print(f"\n{family_name}: Missing base models (need ONNX + LGBM minimum), skipping stacker")
            continue

        print(f"\n{'='*60}")
        print(f"TRAINING STACKER: {family_name.upper()} (family_id={family_id})")
        print(f"{'='*60}")

        # Adjust sequence length
        seq_len = cfg.seq_len
        if family_id in (3, 4) and seq_len == 60:
            seq_len = 120

        # Build dataset
        train_split, val_split, test_split, metadata = build_scaled_dataset_splits(
            cfg.data_csv,
            seq_len=seq_len,
            k=cfg.tb_k,
            vertical_bars=cfg.tb_vertical_bars,
            family_id=family_id,
        )

        X_all = np.concatenate([train_split[0], val_split[0]], axis=0)
        y_all = np.concatenate([train_split[1], val_split[1]], axis=0)
        r_all = np.concatenate([train_split[3], val_split[3]], axis=0)
        X_test = test_split[0]
        r_test = test_split[3]

        X_all_flat = X_all.reshape(len(X_all), -1)
        X_test_flat = X_test.reshape(len(X_test), -1)

        # Load base models
        print("  Loading base models...")
        
        # ONNX model
        sess = ort.InferenceSession(str(onnx_path))
        onnx_input_name = sess.get_inputs()[0].name
        onnx_probs_all = softmax(sess.run(None, {onnx_input_name: X_all.astype(np.float32)})[0], axis=1)
        onnx_probs_test = softmax(sess.run(None, {onnx_input_name: X_test.astype(np.float32)})[0], axis=1)

        # LightGBM
        with open(lgbm_path, "rb") as f:
            lgbm = pickle.load(f)
        lgbm_probs_all = lgbm.predict(X_all_flat)
        lgbm_probs_test = lgbm.predict(X_test_flat)

        # CatBoost (optional)
        catboost_probs_all = None
        catboost_probs_test = None
        if catboost_path.exists():
            with open(catboost_path, "rb") as f:
                catboost = pickle.load(f)
            catboost_probs_all = catboost.predict_proba(X_all_flat)
            catboost_probs_test = catboost.predict_proba(X_test_flat)

        # XGBoost (optional)
        xgboost_probs_all = None
        xgboost_probs_test = None
        if xgboost_path.exists():
            with open(xgboost_path, "rb") as f:
                xgboost = pickle.load(f)
            xgboost_probs_all = xgboost.predict_proba(X_all_flat)
            xgboost_probs_test = xgboost.predict_proba(X_test_flat)

        # Build meta-features using OOF (TimeSeriesSplit)
        n_base = 2  # ONNX + LGBM minimum
        if catboost_probs_all is not None:
            n_base += 1
        if xgboost_probs_all is not None:
            n_base += 1
        n_cols = n_base * 3

        print(f"  Base models: {n_base} (ONNX, LGBM", end="")
        if catboost_probs_all is not None:
            print(", CatBoost", end="")
        if xgboost_probs_all is not None:
            print(", XGBoost", end="")
        print(")")

        meta_X = np.zeros((len(X_all), n_cols), dtype=np.float32)
        tscv = TimeSeriesSplit(n_splits=5, gap=20)
        
        for _, val_idx in tscv.split(X_all):
            col = 0
            meta_X[val_idx, col:col+3] = onnx_probs_all[val_idx]
            col += 3
            meta_X[val_idx, col:col+3] = lgbm_probs_all[val_idx]
            col += 3
            if catboost_probs_all is not None:
                meta_X[val_idx, col:col+3] = catboost_probs_all[val_idx]
                col += 3
            if xgboost_probs_all is not None:
                meta_X[val_idx, col:col+3] = xgboost_probs_all[val_idx]
                col += 3

        # Train Ridge meta-learner (predict BUY probability)
        scaler = StandardScaler().fit(meta_X)
        ridge = Ridge(alpha=1.0).fit(scaler.transform(meta_X), (y_all == 2).astype(float))

        # Test evaluation
        test_parts = [onnx_probs_test, lgbm_probs_test]
        if catboost_probs_test is not None:
            test_parts.append(catboost_probs_test)
        if xgboost_probs_test is not None:
            test_parts.append(xgboost_probs_test)
        test_meta_X = np.hstack(test_parts)
        stack_signal = ridge.predict(scaler.transform(test_meta_X))

        # Save stacker bundle
        bundle = {
            "ridge": ridge,
            "scaler": scaler,
            "family_id": family_id,
            "n_base_models": n_base,
        }
        stacker_output = output_dir / f"{family_name}_stacker.pkl"
        with open(stacker_output, "wb") as f:
            pickle.dump(bundle, f)

        # Test ICs
        print(f"\n  Test IC Comparison:")
        print(f"    ONNX:     {float(spearmanr(onnx_probs_test[:, 2] - onnx_probs_test[:, 0], r_test)[0]):.4f}")
        print(f"    LGBM:     {float(spearmanr(lgbm_probs_test[:, 2] - lgbm_probs_test[:, 0], r_test)[0]):.4f}")
        if catboost_probs_test is not None:
            print(f"    CatBoost: {float(spearmanr(catboost_probs_test[:, 2] - catboost_probs_test[:, 0], r_test)[0]):.4f}")
        if xgboost_probs_test is not None:
            print(f"    XGBoost:  {float(spearmanr(xgboost_probs_test[:, 2] - xgboost_probs_test[:, 0], r_test)[0]):.4f}")
        print(f"    Stacker:  {float(spearmanr(stack_signal, r_test)[0]):.4f}")
        print(f"  Bundle saved: {stacker_output}")

        stacker_results[family_name] = {
            'family_id': family_id,
            'n_base_models': n_base,
            'model_path': str(stacker_output),
            'test_ic_onnx': float(spearmanr(onnx_probs_test[:, 2] - onnx_probs_test[:, 0], r_test)[0]),
            'test_ic_lgbm': float(spearmanr(lgbm_probs_test[:, 2] - lgbm_probs_test[:, 0], r_test)[0]),
            'test_ic_stacker': float(spearmanr(stack_signal, r_test)[0]),
        }
        
        if catboost_probs_test is not None:
            stacker_results[family_name]['test_ic_catboost'] = float(spearmanr(catboost_probs_test[:, 2] - catboost_probs_test[:, 0], r_test)[0])
        if xgboost_probs_test is not None:
            stacker_results[family_name]['test_ic_xgboost'] = float(spearmanr(xgboost_probs_test[:, 2] - xgboost_probs_test[:, 0], r_test)[0])

    # Summary
    print(f"\n{'='*60}")
    print("STACKER TRAINING SUMMARY")
    print(f"{'='*60}")
    for family_name, res in stacker_results.items():
        best_base = max(res['test_ic_onnx'], res['test_ic_lgbm'])
        if 'test_ic_catboost' in res:
            best_base = max(best_base, res['test_ic_catboost'])
        if 'test_ic_xgboost' in res:
            best_base = max(best_base, res['test_ic_xgboost'])
        improvement = res['test_ic_stacker'] - best_base
        print(f"\n  {family_name}:")
        print(f"    Base models: {res['n_base_models']}")
        print(f"    ONNX IC:     {res['test_ic_onnx']:.4f}")
        print(f"    LGBM IC:     {res['test_ic_lgbm']:.4f}")
        if 'test_ic_catboost' in res:
            print(f"    CatBoost IC: {res['test_ic_catboost']:.4f}")
        if 'test_ic_xgboost' in res:
            print(f"    XGBoost IC:  {res['test_ic_xgboost']:.4f}")
        print(f"    Stacker IC:  {res['test_ic_stacker']:.4f} | Δ: {improvement:+.4f} vs best base")

    # Save results
    with open(output_dir / "stacker_training_results.json", "w") as f:
        json.dump(stacker_results, f, indent=2)

else:
    print("Stacker training disabled in config.")

## 7. Universal Stacker (ONNX + Asset Class Models)

In [ ]:
# ============================================================
# Cell 7 — Universal Stacker (Optional)
# ============================================================

universal_stacker_results = {}

# Train a universal stacker combining ONNX model + asset class models
print("="*60)
print("UNIVERSAL STACKER TRAINING")
print("="*60)

try:
    import onnxruntime as ort
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import TimeSeriesSplit
    from scipy.special import softmax
except ImportError as e:
    print(f"\u26a0️  Missing dependencies: {e}")

onnx_path = output_dir / "patchtst.onnx"
if onnx_path.exists():
    # Build universal dataset (family_id=-1, asset_class=-1)
    train_split, val_split, test_split, metadata = build_scaled_dataset_splits(
        cfg.data_csv,
        seq_len=cfg.seq_len,
        k=cfg.tb_k,
        vertical_bars=cfg.tb_vertical_bars,
        family_id=-1,
        asset_class=-1,
    )

    X_all = np.concatenate([train_split[0], val_split[0]], axis=0)
    y_all = np.concatenate([train_split[1], val_split[1]], axis=0)
    r_all = np.concatenate([train_split[3], val_split[3]], axis=0)
    X_test = test_split[0]
    r_test = test_split[3]

    X_all_flat = X_all.reshape(len(X_all), -1)
    X_test_flat = X_test.reshape(len(X_test), -1)

    # Load ONNX model
    sess = ort.InferenceSession(str(onnx_path))
    onnx_input_name = sess.get_inputs()[0].name
    onnx_probs_all = softmax(sess.run(None, {onnx_input_name: X_all.astype(np.float32)})[0], axis=1)
    onnx_probs_test = softmax(sess.run(None, {onnx_input_name: X_test.astype(np.float32)})[0], axis=1)

    # Load asset class models
    asset_models = {}
    for ac_name in ["forex", "metals", "indices", "energies"]:
        for model_type in ["lgbm", "catboost", "xgboost"]:
            pkl_path = output_dir / f"{ac_name}_{model_type}.pkl"
            if pkl_path.exists():
                with open(pkl_path, "rb") as f:
                    asset_models[f"{ac_name}_{model_type}"] = pickle.load(f)
    
    print(f"Loaded {len(asset_models)} asset class models")
    
    # Get predictions from asset models
    asset_probs_all = {}
    asset_probs_test = {}
    for name, model in asset_models.items():
        if hasattr(model, 'predict_proba'):
            asset_probs_all[name] = model.predict_proba(X_all_flat)
            asset_probs_test[name] = model.predict_proba(X_test_flat)
        else:  # LightGBM
            asset_probs_all[name] = model.predict(X_all_flat)
            asset_probs_test[name] = model.predict(X_test_flat)

    # Build meta-features
    n_base = 1 + len(asset_models)  # ONNX + asset models
    n_cols = n_base * 3
    meta_X = np.zeros((len(X_all), n_cols), dtype=np.float32)
    
    tscv = TimeSeriesSplit(n_splits=5, gap=20)
    for _, val_idx in tscv.split(X_all):
        col = 0
        meta_X[val_idx, col:col+3] = onnx_probs_all[val_idx]
        col += 3
        for name in asset_models.keys():
            meta_X[val_idx, col:col+3] = asset_probs_all[name][val_idx]
            col += 3

    # Train Ridge
    scaler = StandardScaler().fit(meta_X)
    ridge = Ridge(alpha=1.0).fit(scaler.transform(meta_X), (y_all == 2).astype(float))

    # Test
    test_parts = [onnx_probs_test]
    for name in asset_models.keys():
        test_parts.append(asset_probs_test[name])
    test_meta_X = np.hstack(test_parts)
    stack_signal = ridge.predict(scaler.transform(test_meta_X))

    # Save
    bundle = {
        "ridge": ridge,
        "scaler": scaler,
        "model_names": ["onnx"] + list(asset_models.keys()),
        "n_base_models": n_base,
    }
    stacker_output = output_dir / "universal_stacker.pkl"
    with open(stacker_output, "wb") as f:
        pickle.dump(bundle, f)

    # Results
    print(f"\nUniversal Stacker Results:")
    print(f"  Base models: {n_base}")
    print(f"  ONNX IC:     {float(spearmanr(onnx_probs_test[:, 2] - onnx_probs_test[:, 0], r_test)[0]):.4f}")
    for name in asset_models.keys():
        ic = float(spearmanr(asset_probs_test[name][:, 2] - asset_probs_test[name][:, 0], r_test)[0])
        print(f"  {name:20s} IC: {ic:.4f}")
    print(f"  Stacker IC:  {float(spearmanr(stack_signal, r_test)[0]):.4f}")
    print(f"  Saved: {stacker_output}")

    universal_stacker_results = {
        'n_base_models': n_base,
        'model_names': ["onnx"] + list(asset_models.keys()),
        'model_path': str(stacker_output),
        'test_ic_onnx': float(spearmanr(onnx_probs_test[:, 2] - onnx_probs_test[:, 0], r_test)[0]),
        'test_ic_stacker': float(spearmanr(stack_signal, r_test)[0]),
    }
    for name in asset_models.keys():
        universal_stacker_results[f'test_ic_{name}'] = float(spearmanr(asset_probs_test[name][:, 2] - asset_probs_test[name][:, 0], r_test)[0])

    with open(output_dir / "universal_stacker_results.json", "w") as f:
        json.dump(universal_stacker_results, f, indent=2)
else:
    print(f"\u26a0️  Universal ONNX model not found at {onnx_path}, skipping universal stacker")

## 8. Final Summary & Model Registry

In [ ]:
# ============================================================
# Cell 8 — Final Summary & Model Registry
# ============================================================

print("="*80)
print("COMPREHENSIVE TRAINING SUMMARY")
print("="*80)

# Collect all model files
model_files = list(output_dir.glob("*.onnx")) + list(output_dir.glob("*.pkl")) + list(output_dir.glob("*.bin"))

print(f"\nOutput directory: {output_dir}")
print(f"Total model files: {len(model_files)}")

for f in sorted(model_files):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s} {size_mb:>8.2f} MB")

# PyTorch results
if 'results' in locals() and results:
    print(f"\n{'='*80}")
    print("PYTORCH MODELS (ONNX)")
    print(f"{'='*80}")
    for model_type, res in results.items():
        status = "✅ DEPLOYED" if res.get('deploy_ok') else "❌ REJECTED"
        print(f"  {model_type:12s} | Val IC: {res['val_ic']:.4f} | Test IC: {res['test_ic']:.4f} | CPCV Sharpe: {res['cpcv']['mean_sr']:.4f} | {status}")

# Asset class results
if 'asset_class_results' in locals() and asset_class_results:
    print(f"\n{'='*80}")
    print("ASSET CLASS MODELS (Tree-Based)")
    print(f"{'='*80}")
    for ac_name, models in asset_class_results.items():
        print(f"\n  {ac_name.upper()}:")
        for model_type, res in models.items():
            print(f"    {model_type:10s} | Test IC: {res['test_ic']:.4f} | {Path(res['model_path']).name}")

# Deriv family results
if 'deriv_results' in locals() and deriv_results:
    print(f"\n{'='*80}")
    print("DERIV FAMILY MODELS (18 Families)")
    print(f"{'='*80}")
    for family_name, res in deriv_results.items():
        print(f"\n  {family_name} (id={res['family_id']}, seq_len={res['seq_len']}, n={res['dataset_size']:,}):")
        for model_type, mres in res['models'].items():
            print(f"    {model_type:10s} | Test IC: {mres['test_ic']:.4f}")

# Stacker results
if 'stacker_results' in locals() and stacker_results:
    print(f"\n{'='*80}")
    print("FAMILY STACKERS")
    print(f"{'='*80}")
    for family_name, res in stacker_results.items():
        best_base = max(res['test_ic_onnx'], res['test_ic_lgbm'])
        if 'test_ic_catboost' in res:
            best_base = max(best_base, res['test_ic_catboost'])
        if 'test_ic_xgboost' in res:
            best_base = max(best_base, res['test_ic_xgboost'])
        improvement = res['test_ic_stacker'] - best_base
        print(f"  {family_name:20s} | Stacker IC: {res['test_ic_stacker']:.4f} | Best base: {best_base:.4f} | Δ: {improvement:+.4f}")

# Universal stacker
if 'universal_stacker_results' in locals() and universal_stacker_results:
    print(f"\n{'='*80}")
    print("UNIVERSAL STACKER")
    print(f"{'='*80}")
    res = universal_stacker_results
    best_base = max(v for k, v in res.items() if k.startswith('test_ic_'))
    improvement = res['test_ic_stacker'] - best_base
    print(f"  Base models: {res['n_base_models']}")
    print(f"  ONNX IC:     {res['test_ic_onnx']:.4f}")
    for name in res['model_names'][1:]:
        print(f"  {name:20s} IC: {res[f'test_ic_{name}']:.4f}")
    print(f"  Stacker IC:  {res['test_ic_stacker']:.4f} | Δ: {improvement:+.4f}")

# Create model registry
registry = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "data_file": cfg.data_csv,
    "config": {
        "seq_len": cfg.seq_len,
        "tb_k": cfg.tb_k,
        "tb_vertical_bars": cfg.tb_vertical_bars,
        "epochs": cfg.epochs,
        "batch_size": cfg.batch_size,
        "lr": cfg.lr,
        "device": cfg.device,
    },
    "models": {},
}

# Add PyTorch models
if 'results' in locals() and results:
    for model_type, res in results.items():
        if res.get('model_path'):
            registry["models"][f"pytorch_{model_type}"] = {
                "type": "pytorch_onnx",
                "model_type": model_type,
                "path": res['model_path'],
                "val_ic": res['val_ic'],
                "test_ic": res['test_ic'],
                "cpcv_sharpe": res['cpcv']['mean_sr'],
                "deployed": res['deploy_ok'],
            }

# Add asset class models
if 'asset_class_results' in locals() and asset_class_results:
    for ac_name, models in asset_class_results.items():
        for model_type, res in models.items():
            registry["models"][f"asset_class_{ac_name}_{model_type}"] = {
                "type": "tree_based",
                "asset_class": ac_name,
                "model_type": model_type,
                "path": res['model_path'],
                "test_ic": res['test_ic'],
            }

# Add Deriv family models
if 'deriv_results' in locals() and deriv_results:
    for family_name, res in deriv_results.items():
        for model_type, mres in res['models'].items():
            registry["models"][f"deriv_{family_name}_{model_type}"] = {
                "type": "tree_based",
                "family_id": res['family_id'],
                "family_name": family_name,
                "model_type": model_type,
                "path": mres['model_path'],
                "test_ic": mres['test_ic'],
                "seq_len": res['seq_len'],
            }

# Add stackers
if 'stacker_results' in locals() and stacker_results:
    for family_name, res in stacker_results.items():
        registry["models"][f"stacker_{family_name}"] = {
            "type": "stacker",
            "family_id": res['family_id'],
            "family_name": family_name,
            "path": res['model_path'],
            "n_base_models": res['n_base_models'],
            "test_ic_stacker": res['test_ic_stacker'],
        }

if 'universal_stacker_results' in locals() and universal_stacker_results:
    registry["models"]["universal_stacker"] = {
        "type": "stacker",
        "scope": "universal",
        "path": universal_stacker_results['model_path'],
        "n_base_models": universal_stacker_results['n_base_models'],
        "test_ic_stacker": universal_stacker_results['test_ic_stacker'],
    }

# Save registry
registry_path = output_dir / "model_registry.json"
with open(registry_path, "w") as f:
    json.dump(registry, f, indent=2, default=str)

print(f"\n{'='*80}")
print(f"MODEL REGISTRY SAVED: {registry_path}")
print(f"Total models registered: {len(registry['models'])}")
print(f"{'='*80}")

## 9. Quick Inference Test (Verify Exports)

In [ ]:
# ============================================================
# Cell 9 — Quick Inference Test
# ============================================================

print("="*60)
print("INFERENCE VERIFICATION")
print("="*60)

# Test ONNX models with onnxruntime
try:
    import onnxruntime as ort
    
    # Test universal ONNX model
    onnx_path = output_dir / "patchtst.onnx"
    if onnx_path.exists():
        print(f"\nTesting: {onnx_path.name}")
        sess = ort.InferenceSession(str(onnx_path))
        input_name = sess.get_inputs()[0].name
        output_name = sess.get_outputs()[0].name
        
        # Create dummy input
        dummy = np.random.randn(1, 60, 57).astype(np.float32)
        output = sess.run([output_name], {input_name: dummy})[0]
        
        print(f"  Input shape:  {dummy.shape}")
        print(f"  Output shape: {output.shape}")
        print(f"  Output logits: {output[0]}")
        pred_class = np.argmax(output[0])
        print(f"  Predicted:     {pred_class} ({['SELL','NEUTRAL','BUY'][pred_class]})")
        print(f"  ✅ ONNX inference OK")
    else:
        print(f"\n⚠️  Universal ONNX model not found: {onnx_path}")
        
    # Test a family ONNX if exists
    for family_name in FAMILY_NAMES[:3]:
        family_onnx = output_dir / f"{family_name}.onnx"
        if family_onnx.exists():
            print(f"\nTesting family ONNX: {family_onnx.name}")
            sess = ort.InferenceSession(str(family_onnx))
            input_name = sess.get_inputs()[0].name
            output_name = sess.get_outputs()[0].name
            dummy = np.random.randn(1, 60, 83).astype(np.float32)
            output = sess.run([output_name], {input_name: dummy})[0]
            print(f"  Input shape: {dummy.shape}, Output: {output.shape}")
            print(f"  ✅ Family ONNX inference OK")
            break
    
    # Test a tree-based model
    test_model = None
    for ac in ["forex", "metals", "indices", "energies"]:
        for mt in ["lgbm", "catboost", "xgboost"]:
            p = output_dir / f"{ac}_{mt}.pkl"
            if p.exists():
                test_model = p
                break
        if test_model:
            break
    
    if test_model:
        print(f"\nTesting tree model: {test_model.name}")
        with open(test_model, "rb") as f:
            model = pickle.load(f)
        
        dummy = np.random.randn(1, 57*60).astype(np.float32)
        if hasattr(model, 'predict_proba'):
            preds = model.predict_proba(dummy)
        else:
            preds = model.predict(dummy)
        print(f"  Output shape: {preds.shape}")
        print(f"  Output probs: {preds[0]}")
        pred_class = np.argmax(preds[0])
        print(f"  Predicted:    {pred_class} ({['SELL','NEUTRAL','BUY'][pred_class]})")
        print(f"  ✅ Tree model inference OK")
    else:
        print(f"\n⚠️  No tree-based model found for testing")
        
except ImportError:
    print("\u26a0️  onnxruntime not installed - skipping ONNX inference test")
except Exception as e:
    print(f"Error testing ONNX model: {e}")